# Part 3: Topic graph

In Part 2, the entity graph linked named things, such as a batch and its farms, through typed relationships. Each relationship stated a fact that you could trace back to source text. A **topic graph** answers a different question: what does this content cover, and where should I look?

**🎯 Mission**
- See how similar chunks are clustered into topics and subtopics, and how much of the content the topics cover.
- Compare non-diversified vector search with diversified vector search, which uses the topic graph to spread results across topics.
- Walk the topic graph with GQL, from broad topics to their subtopics.
- Investigate a customer complaint: ask a topic knowledge base where to look, then answer the complaint with a topic-aware knowledge source that cites the PDF evidence.

Before opening this notebook, you browsed the topic graph on the graph explorer's **Topics** tab. This notebook reads the same data directly, so you don't need to remember the details.

## How the topic graph is built

When the lab data was prepared, the Foundry IQ plugin built the topic graph from chunk embeddings in two stages: it clustered the chunks, then turned the clusters into topics.

<img src="../../img/diagrams/notebook-03-topics-1-chunks.svg" alt="Step 1, cluster chunks based on embeddings. Each chunk is linked to its most similar chunks. The clustering method groups strongly linked chunks into a broad cluster, then splits it into tighter clusters, A and B. It also finds a separate cluster, C. One chunk has no strong links, so it isn't clustered. Chunks are text from the original PDFs, not entities or extracted facts." width="800">

First, each chunk is linked to its most similar chunks by comparing their embeddings. A clustering method then groups chunks that are more strongly linked to each other than to the rest. It works hierarchically: it finds broad clusters, then splits a cluster that's too large into tighter clusters. A chunk without strong enough links to other chunks isn't clustered. Similarity isn't a fact: two similar chunks don't state anything verified about each other.

<img src="../../img/diagrams/notebook-03-topics-2-topics.svg" alt="Step 2, build topics from clusters of chunks. Each level of clusters becomes a level of topics. The broad cluster becomes a broad topic at level 0, and the tighter clusters it was split into, A and B, become subtopics at level 1. The separate cluster, C, becomes a broad topic with no subtopics. Each terminal topic stores a centroid." width="800">

Next, each level of clusters becomes a level of **topics**:

- Each broad cluster becomes a **broad topic** at level 0.
- When a broad cluster is split, each tighter cluster becomes a **subtopic** at level 1, linked to its broad topic.
- A **terminal topic** has no subtopics. It's either a subtopic or a broad topic that wasn't split.
- Each topic has a **title** and **description** generated from a sample of its chunks. They help you decide where to look, but they don't define the topic.
- A **centroid** is a vector at the center of a topic's chunk embeddings. Only terminal topics store one. Because a centroid is comparable to chunk embeddings, you can use it to find a chunk's nearest topic, or a topic's nearest chunks.

This lab's topic graph has four broad topics. Two of them were split into two subtopics each, so there are six terminal topics: four subtopics and two broad topics.

<img src="../../img/diagrams/notebook-03-topic-graph.svg" alt="Step 3, navigate the topic graph, then read the evidence. The topic index stores the topic graph as facts. The TopicGraph topicgraph-topic points to each broad topic at level 0 through HasRootTopic. A broad topic points to each subtopic at level 1 through HasSubTopic. Each terminal topic points to its centroid Vector through HasCentroidVector. A vector search with a centroid over cocoarynth-chunks returns the nearest PDF chunks: examples for browsing, not stored membership. Topics guide where to look. Source chunks provide the evidence." width="800">

The topic index stores the topic graph as facts. *HasRootTopic* links the graph to its broad topics, *HasSubTopic* links a broad topic to its subtopics, and *HasCentroidVector* links a terminal topic to its centroid. The ontology file, **cocoarynth-sitemap.owl**, calls this structure a sitemap.

The topic graph shows where to look, but only the source records can confirm what happened. Topics also don't cover every chunk, so a missing topic doesn't prove that the documents never mention a subject.

## Step 1: Load your settings and connect

You'll query two indexes directly:

- `cocoarynth-topics` stores the topic graph: topic titles, descriptions, links, and centroids.
- `cocoarynth-chunks` stores the PDF chunks and their embeddings.

Run this notebook from the repository root or `src/notebooks`.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
API_VERSION = "2026-11-01-preview"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

In [ ]:
import json
import textwrap
from pathlib import Path

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

TOPICS_INDEX = "cocoarynth-topics"
CHUNKS_INDEX = "cocoarynth-chunks"
TOPICS_KNOWLEDGE_SOURCE = f"{TOPICS_INDEX}-knowledge-source"

topics_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, TOPICS_INDEX, search_credential, api_version=API_VERSION
)
chunks_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, CHUNKS_INDEX, search_credential, api_version=API_VERSION
)
print(f"{TOPICS_INDEX}: {topics_client.get_document_count()} topic graph facts")
print(f"{CHUNKS_INDEX}: {chunks_client.get_document_count()} PDF chunks")

## Look inside the topic graph

Steps 2–7 read the topic graph directly, so you can see the data behind the explorer's **Topics** tab: how much content the topics cover, how a centroid relates to its chunks, how search can use topics to diversify results, and how topics link to subtopics.

### Step 2: Check topic coverage

The [restore manifest](../../data/graph-data/manifest.json) records how many chunks the build assigned to topics, and at which level. Run the cell, then compare the numbers with the **Topics** tab.

You should see 97 of the 106 chunks assigned to topics. Every assigned chunk belongs to one of four broad topics, and 61 of them also belong to a subtopic. The other 9 chunks aren't in any topic.

In [ ]:
metadata_file = Path("data") / "graph-data" / "manifest.json"
if not metadata_file.is_file():
    metadata_file = Path("..") / ".." / metadata_file
build_metadata = json.loads(metadata_file.read_text(encoding="utf-8"))["topics"]
assert build_metadata["schema_version"] == "topicgraph-release-v2"

chunks_by_level = build_metadata["assigned_chunks_by_level"]
print(f"Schema version:              {build_metadata['schema_version']}")
print(f"PDF chunks in the lab:       {build_metadata['source_document_count']}")
print(f"Chunks assigned to topics:   {build_metadata['clustered_source_document_count']} "
      f"({build_metadata['node_coverage']:.1%} coverage)")
print(f"Chunks not in any topic:     {build_metadata['unassigned_source_document_count']}")
print(f"Chunks in a broad topic:     {chunks_by_level['0']} (level 0)")
print(f"Chunks also in a subtopic:   {chunks_by_level['1']} (level 1)")
print(f"Topics:                      {build_metadata['topic_count']} "
      f"({build_metadata['terminal_topic_count']} terminal, each with a centroid)")

### Step 3: Pick a chunk to start from

Steps 3–6 follow one chunk. You'll find its nearest chunks across the whole index, then its nearest chunks inside its nearest topic, and finally compare non-diversified and diversified vector search for it.

The cell finds a chunk that mentions Madagascar bean lot MG26-L02 and reuses its stored embedding. The question is "what's near this chunk?", so no embedding model is called.

In [ ]:
start_chunk = list(chunks_client.search(
    search_text='"MG26-L02"',
    search_fields=["chunk"],
    select=["chunk_id", "title", "blob_path", "chunk", "text_vector"],
    top=1,
))[0]
query_vector = start_chunk["text_vector"]
print(f"Starting chunk: {start_chunk['title']} ({start_chunk['blob_path']})")
print(textwrap.indent(textwrap.shorten(start_chunk["chunk"], 400), "    "))

### Step 4: Find the nearest chunks directly

Run a vector search over all chunks with the starting chunk's embedding. The first result is usually the starting chunk itself.

In [ ]:
direct = list(chunks_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

print("Nearest chunks across the whole index:")
for rank, result in enumerate(direct, 1):
    print(f"  {rank}. {result['title']} ({result['blob_path']})")

### Step 5: Find the nearest topic, then search its centroid-nearest candidates

Now route the same search through a topic. The cell runs three vector searches:

1. **Nearest topic**: search the centroids (the `HasCentroidVector` facts) for the one closest to the starting chunk. Only terminal topics have centroids, so the nearest topic is always a terminal topic.
2. **Centroid-nearest candidates**: search `cocoarynth-chunks` with that centroid. The topic's size is its *proportion* of the 97 assigned chunks, so the cell keeps that many nearest chunks (at most 1,000). Like the chunk list on the explorer's **Topics** tab, this candidate set approximates the topic's core; the topic graph doesn't store build-time chunk membership.
3. **Search the candidates**: rerun the starting chunk's search, filtered to that candidate set.

The final check shows whether the starting chunk appears in this centroid-nearest candidate set. It does not establish whether the build assigned that chunk to the topic.

This exercise shows how a centroid relates to its chunks. It isn't a recommended way to search: it adds two queries and can miss relevant chunks. Step 6 shows the built-in way to use topics in vector search.

In [ ]:
nearest_centroid = list(topics_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="object_vector", k_nearest_neighbors=1, exhaustive=True)],
    filter="predicate eq 'HasCentroidVector'",
    select=["subject_id", "object_vector"],
    top=1,
))[0]
NEAREST_TOPIC_ID = nearest_centroid["subject_id"]
nearest_topic = {
    fact["predicate"]: fact["object_string"] if fact["object_string"] is not None else fact["object_double"]
    for fact in topics_client.search(
        filter=f"subject_id eq '{NEAREST_TOPIC_ID}'",
        select=["predicate", "object_string", "object_double"],
    )
    if fact["predicate"].startswith("prop:")
}
topic_size = min(round(nearest_topic["prop:proportion"] * build_metadata["clustered_source_document_count"]), 1000)

candidate_ids = [
    member["chunk_id"]
    for member in chunks_client.search(
        vector_queries=[VectorizedQuery(vector=nearest_centroid["object_vector"], fields="text_vector", k_nearest_neighbors=topic_size, exhaustive=True)],
        select=["chunk_id"],
        top=topic_size,
    )
]
routed = list(chunks_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    filter=f"search.in(chunk_id, '{','.join(candidate_ids)}', ',')",
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

print(f"Nearest topic:  {nearest_topic['prop:title']} ({NEAREST_TOPIC_ID}, level {nearest_topic['prop:level']:.0f})")
print(f"Candidate set:  {len(candidate_ids)} chunks nearest the topic centroid")
print(f"Starting chunk appears in the candidate set: {start_chunk['chunk_id'] in candidate_ids}\n")
print("Nearest chunks within the candidate set:")
for rank, result in enumerate(routed, 1):
    print(f"  {rank}. {result['title']} ({result['blob_path']})")

### Step 6: Compare non-diversified and diversified vector search

Foundry IQ can use the topic graph inside a vector query. A **non-diversified** search returns the chunks most similar to the query vector, even if they all come from one topic. A **diversified** search spreads its results across topics, so one dominant topic doesn't fill every slot.

![Diagram: the same query vector feeds two result lists. The non-diversified list is ranked by similarity score, and most of its chunks are nearest topic A. The diversified list picks from topics A, B, C, and D in turns. It keeps each chunk's original score, so its scores aren't in descending order, and two close topic A matches are replaced by chunks nearest other topics. Each chunk's nearest topic is found at query time.](../../img/diagrams/notebook-03-diversified-search.svg)

To turn on diversified search, the cell first connects the chunk index to the topic graph:

- It adds a *topic graph profile* to `cocoarynth-chunks` that points to the `cocoarynth-topics` index, and assigns the profile to the `text_vector` field. A profile can only be added to an index that already exists, so the cell updates the index.
- The profile's *defaultKind* is *none*, so queries that don't ask for diversity behave as before.

Then it runs the same vector query twice with the starting chunk's embedding, changing only the query's *topicGraph* setting:

- **Non-diversified search** (*kind*: *none*): results are ranked by similarity alone, as in Step 4.
- **Diversified search** (*kind*: *diverse*): the search service gathers a larger pool of similar chunks, assigns each one to its nearest topic centroid, and then picks results from the topics in rounds. The *diversityFactor*, from 0 to 1, sets how far another topic's best chunk may score below the leading topic's chunks and still be picked in a round. The cell uses 0.5 to make the effect easy to see; the default is 0.2.

For each result, the cell prints its similarity score, its PDF, and its nearest topic, then counts the topics that each list covers. Diversified results keep their original similarity scores but are returned in the order they were picked, so their scores aren't always in descending order.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import VectorSearchTopicGraphProfile
from azure.search.documents.models import DiverseVectorQueryTopicGraph, NoneVectorQueryTopicGraph

TOPIC_GRAPH_PROFILE = "cocoarynth-topic-graph"
index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)
chunks_index = index_client.get_index(CHUNKS_INDEX)
chunks_index.vector_search.topic_graph_profiles = [VectorSearchTopicGraphProfile(
    name=TOPIC_GRAPH_PROFILE, index_name=TOPICS_INDEX, default_kind="none",
)]
next(field for field in chunks_index.fields if field.name == "text_vector").topic_graph_profile = TOPIC_GRAPH_PROFILE
index_client.create_or_update_index(chunks_index)
print(f"Topic graph profile '{TOPIC_GRAPH_PROFILE}': {CHUNKS_INDEX}.text_vector uses {TOPICS_INDEX}\n")

topic_titles = {
    fact["subject_id"]: fact["object_string"]
    for fact in topics_client.search(
        filter="predicate eq 'prop:title'", select=["subject_id", "object_string"], top=50,
    )
}


def nearest_topic_title(vector):
    centroid = list(topics_client.search(
        vector_queries=[VectorizedQuery(vector=vector, fields="object_vector", k_nearest_neighbors=1, exhaustive=True)],
        filter="predicate eq 'HasCentroidVector'",
        select=["subject_id"],
        top=1,
    ))[0]
    return topic_titles[centroid["subject_id"]]


def compare_search(label, topic_graph):
    results = list(chunks_client.search(
        vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, topic_graph=topic_graph)],
        select=["chunk_id", "blob_path", "text_vector"],
        top=8,
    ))
    topics = [nearest_topic_title(result["text_vector"]) for result in results]
    print(f"{label}: {len(set(topics))} topics")
    for rank, (result, topic) in enumerate(zip(results, topics), 1):
        print(f"  {rank}. {result['@search.score']:.3f}  {result['blob_path'].rsplit('/', 1)[-1]:<36} {topic}")
    print()
    return results, topics


non_diversified, non_diversified_topics = compare_search(
    "Non-diversified search (kind: none)", NoneVectorQueryTopicGraph(),
)
diversified, diversified_topics = compare_search(
    "Diversified search (kind: diverse, diversityFactor: 0.5)", DiverseVectorQueryTopicGraph(diversity_factor=0.5),
)
non_diversified_ids = {result["chunk_id"] for result in non_diversified}
diversified_ids = {result["chunk_id"] for result in diversified}
print(f"In both lists:             {len(non_diversified_ids & diversified_ids)}")
print(f"Only in diversified list:  {len(diversified_ids - non_diversified_ids)}")

### Step 7: Walk the topic graph with GQL

The topic index's graph profile exposes three relationships: `TopicGraph → HasRootTopic → Topic`, `Topic → HasSubTopic → Topic`, and `Topic → HasCentroidVector → Vector`.

The cell runs two GQL queries. The first follows *HasRootTopic* to the broad topics, and the second follows *HasSubTopic* to every subtopic. It prints the result as a tree, like the explorer's **Topics** tab. For each terminal topic, a separate centroid search of `cocoarynth-chunks` lists up to two example PDFs from the chunks nearest its centroid.

Compare each topic's title with its example PDFs. Does the title describe them well?

In [ ]:
ROOT_TOPICS_GQL = """
MATCH (topic_graph:`TopicGraph`)-[:`HasRootTopic`]->(topic:`Topic`)
RETURN topic.id AS topic_id, topic.title AS title, topic.description AS description
LIMIT 50
"""
SUBTOPICS_GQL = """
MATCH (parent:`Topic`)-[:`HasSubTopic`]->(topic:`Topic`)
RETURN parent.id AS parent_id, topic.id AS topic_id, topic.title AS title, topic.description AS description
LIMIT 50
"""
root_topics = topics_client.graph(gql=ROOT_TOPICS_GQL, include_statistics=True).value
subtopics = topics_client.graph(gql=SUBTOPICS_GQL, include_statistics=True).value
centroids = {
    fact["subject_id"]: fact["object_vector"]
    for fact in topics_client.search(
        filter="predicate eq 'HasCentroidVector'", select=["subject_id", "object_vector"], top=50,
    )
}
print(f"{len(root_topics)} broad topics, {len(subtopics)} subtopics, {len(centroids)} terminal topics with centroids\n")


def print_topic(topic, prefix, detail_prefix):
    is_terminal = topic["topic_id"] in centroids
    print(f"{prefix}{topic['title']} ({topic['topic_id']}{', terminal' if is_terminal else ''})")
    print(f"{detail_prefix}Description: {textwrap.shorten(topic['description'], 160)}")
    if is_terminal:
        nearest_chunks = chunks_client.search(
            vector_queries=[VectorizedQuery(vector=centroids[topic["topic_id"]], fields="text_vector", k_nearest_neighbors=5, exhaustive=True)],
            select=["blob_path"], top=5,
        )
        example_pdfs = list(dict.fromkeys(chunk["blob_path"] for chunk in nearest_chunks))[:2]
        print(f"{detail_prefix}Example PDFs: {', '.join(example_pdfs)}")


for root in sorted(root_topics, key=lambda topic: topic["topic_id"]):
    print_topic(root, "", "  ")
    for subtopic in sorted((row for row in subtopics if row["parent_id"] == root["topic_id"]), key=lambda topic: topic["topic_id"]):
        print_topic(subtopic, "  └─ ", "       ")
    print()

## Use the topic graph in retrieval

Now make the same `retrieve()` calls that an app would make. Step 8 asks a topic knowledge base where to look. Step 9 answers the complaint with a topic-aware knowledge source, which searches the PDF chunks with diversified search.

### Step 8: Ask the topic knowledge base where to look

Larkspur Market has complained about grey, dusty-looking Ghana bars. Before deciding what to do, a quality lead needs to find records about the batch, its delivery, and the response to the customer.

`cocoarynth-topics-knowledge-source` connects a knowledge base to the topic index. Its `graphOntology` holds the [sitemap ontology](../../data/graph-data/cocoarynth-sitemap.owl), converted to inline JSON-LD, which describes the topic graph's types and relationships to graph search.

![Diagram: cocoarynth-topics-knowledge-base uses cocoarynth-topics-knowledge-source, whose graphOntology is cocoarynth-sitemap.owl converted to inline JSON-LD. The knowledge source searches the cocoarynth-topics index: 54 facts about 8 topics, including 6 centroid vectors, linked by HasRootTopic and HasSubTopic. A separate centroid search finds nearby chunks in cocoarynth-chunks.](../../img/diagrams/notebook-03.svg)

The cell creates a knowledge base over this source and asks where to look. Its answer instructions limit the model to the returned topic titles and descriptions. The planner can use graph search or text search. When it uses graph search, the cell prints the generated GQL, because the request sets *graphParameters* to `{"includeQuery": True}`. If no `graphSearch` step appears, the planner used text search this time.

Use the suggested topics to orient the investigation. In Step 9, you'll check the original records.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model, KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort, KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

topic_source = index_client.get_knowledge_source(TOPICS_KNOWLEDGE_SOURCE).as_dict()
topic_ontology = topic_source["searchIndexParameters"].get("graphOntology", {})
print(f"Knowledge source: {TOPICS_KNOWLEDGE_SOURCE}")
print(f"Inline sitemap ontology: {topic_ontology['name']}")

KNOWLEDGE_BASE_MODELS = [
    KnowledgeBaseMicrosoftRetriever1Model(),
    KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
        resource_url=AZURE_OPENAI_ENDPOINT, deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
        model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME, api_key=AZURE_OPENAI_KEY,
    )),
]
TOPICS_KB_NAME = "cocoarynth-topics-knowledge-base"
index_client.create_or_update_knowledge_base(KnowledgeBase(
    name=TOPICS_KB_NAME,
    description="Suggest where to look in the Cocoarynth records by searching the topic graph",
    knowledge_sources=[KnowledgeSourceReference(name=TOPICS_KNOWLEDGE_SOURCE)],
    models=KNOWLEDGE_BASE_MODELS,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Use returned topic titles and descriptions only and cite reference IDs. Do not invent PDF membership or business facts.",
))
print(f"Knowledge base '{TOPICS_KB_NAME}' created or updated.\n")


def investigation_request(question, knowledge_source_names, include_graph_query=False):
    source_params = [SearchIndexKnowledgeSourceParams(
        knowledge_source_name=name, include_references=True,
        include_reference_source_data=True, always_query_source=True,
    ) for name in knowledge_source_names]
    if include_graph_query:
        for source_param in source_params:
            if source_param.knowledge_source_name == TOPICS_KNOWLEDGE_SOURCE:
                source_param["graphParameters"] = {"includeQuery": True}
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=question)])],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        include_activity=True,
        knowledge_source_params=source_params,
    )


DISCOVERY_QUESTION = (
    "I'm investigating Larkspur Market's complaint about grey, dusty Ghana bars. "
    "Which areas of our records should I explore to assess the batch, "
    "what happened during delivery, and our response to the customer?"
)
topics_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=TOPICS_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
topic_result = topics_kb_client.retrieve(investigation_request(
    DISCOVERY_QUESTION, [TOPICS_KNOWLEDGE_SOURCE],
    include_graph_query=True,
))
print(f"Question: {DISCOVERY_QUESTION}\n")
print("Answer (from topic titles and descriptions):")
display(Markdown(topic_result.response[0].content[0].text))

print("Activity:")
for activity in topic_result.activity or []:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")
    if step["type"] == "graphSearch":
        arguments = step.get("graphSearchArguments") or {}
        print(f"    Request:       {arguments.get('naturalLanguageQuery')}")
        print(f"    Generated GQL: {arguments.get('gql')}")

### Step 9: Answer the complaint with a topic-aware knowledge source

Now ask the full question: was batch GH26-0706 itself bad, or only Larkspur's delivery? Should other stores that received the batch pull their bars, and what's already been done for Larkspur?

The answer depends on records from different areas: quality checks of retained samples, delivery records, and the customer's credit or replacement. A non-diversified search can fill its results with chunks from one area. A **topic-aware knowledge source** uses diversified search, so its results can span several topics.

![Diagram: cocoarynth-topic-aware-knowledge-base uses cocoarynth-topic-aware-knowledge-source, whose topicGraph kind is diverse. The knowledge source searches the cocoarynth-chunks index. Its text_vector field uses the cocoarynth-topic-graph profile, which points to the topic centroids in the cocoarynth-topics index.](../../img/diagrams/notebook-03-topic-aware.svg)

The cell creates `cocoarynth-topic-aware-knowledge-source` over `cocoarynth-chunks`:

- *topicGraph* is set to *diverse*, so the vector queries that the knowledge source sends use diversified search. It doesn't set *diversityFactor*, so the default of 0.2 applies.
- *searchFields* include `text_vector`, the field with the topic graph profile from Step 6. A *topicGraph* setting needs at least one search field with a topic graph profile.
- *sourceDataFields* return each chunk's title, PDF path, and text, so you can check the evidence.

The cell then creates a knowledge base over the new source and asks the question. This knowledge base searches PDF chunks, not topic titles, so its answer can cite the original records. Use the topic suggestions from Step 8 to decide which records to look for in the answer.

Read the answer and its references. Which records describe retained samples, which describe delivery conditions, and which record the customer's credit or replacement? Check the cited evidence before deciding what the findings mean for other stores.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference, SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters,
)

TOPIC_AWARE_KNOWLEDGE_SOURCE = "cocoarynth-topic-aware-knowledge-source"
index_client.create_or_update_knowledge_source(SearchIndexKnowledgeSource(
    name=TOPIC_AWARE_KNOWLEDGE_SOURCE,
    description="All Cocoarynth PDF chunks, retrieved with topic-diversified vector search",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=CHUNKS_INDEX,
        semantic_configuration_name="semantic-configuration",
        search_fields=[SearchIndexFieldReference(name=name) for name in ("chunk", "text_vector")],
        source_data_fields=[
            SearchIndexFieldReference(name=name) for name in ("chunk_id", "title", "blob_path", "chunk")
        ],
        topic_graph=DiverseVectorQueryTopicGraph(),
    ),
))
print(f"Knowledge source '{TOPIC_AWARE_KNOWLEDGE_SOURCE}' created or updated.")

TOPIC_AWARE_KB_NAME = "cocoarynth-topic-aware-knowledge-base"
index_client.create_or_update_knowledge_base(KnowledgeBase(
    name=TOPIC_AWARE_KB_NAME,
    description="Answer questions about Cocoarynth from PDF chunks retrieved with topic-diversified search",
    knowledge_sources=[KnowledgeSourceReference(name=TOPIC_AWARE_KNOWLEDGE_SOURCE)],
    models=KNOWLEDGE_BASE_MODELS,
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Answer only from the returned PDF chunks and cite reference IDs. Say which parts of the question the chunks don't answer.",
))
print(f"Knowledge base '{TOPIC_AWARE_KB_NAME}' created or updated.\n")

COMPLAINT_QUESTION = (
    "Larkspur Market received grey, dusty Ghana bars from batch GH26-0706. "
    "Was the batch itself bad, or just that delivery? "
    "Should other stores that received this batch pull their bars, "
    "and what have we already done for Larkspur?"
)
topic_aware_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=TOPIC_AWARE_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
document_result = topic_aware_client.retrieve(investigation_request(
    COMPLAINT_QUESTION, [TOPIC_AWARE_KNOWLEDGE_SOURCE],
))
print(f"Question: {COMPLAINT_QUESTION}\n")
print("Answer (from PDF chunks retrieved with diversified search):")
display(Markdown(document_result.response[0].content[0].text))

print("Activity:")
for activity in document_result.activity or []:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<40} results: {step.get('count', '-')}")

print("\nReferences (PDF evidence):")
for reference in document_result.references or []:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "blob_path" in source_data:
        print(f"    PDF: {source_data['blob_path']}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 400), "      "))

## Choose the right approach for the question

Your two `retrieve()` calls drew on different evidence. The topic knowledge base searched the topic graph, so its answer could point you to areas of the records. The topic-aware knowledge source searched the PDF chunks with diversified search, so its answer could cite records from several areas.

| Need | Good starting point | Tradeoff |
| --- | --- | --- |
| Ground a direct question in original documents | Standard knowledge base | The model plans queries, picks sources, and writes a cited answer |
| Retrieve chunks quickly | microsoft-retriever-1 | Faster planning without a chat completion model, but it searches every source and returns chunks, not an answer |
| Follow explicit relationships across records | Entity graph and GQL | Inspectable joins and provenance, but depends on the quality of the stored relationships |
| Find out what the content covers and where to look | Topic graph and GQL | Browse broad topics and subtopics of similar chunks, but topics guide where to look; they don't state facts |
| Answer a question whose evidence spans several areas | Topic-aware knowledge source | Diversified search spreads results across topics, but it can replace some of the closest matches with chunks from other topics |

Think about the three questions from this lab: the organic seal, the farms behind batch EC26-0811, and Larkspur's complaint. Which approach would you start with for each, and why?

## ✅ Mission complete

**What you did:**

* ✓ **Topic coverage**: Checked how many chunks the topic graph covers, and at which level.
* ✓ **Topic centroids**: Compared a chunk's nearest chunks across the whole index with its nearest chunks inside its nearest topic.
* ✓ **Diversified search**: Connected the chunk index to the topic graph, then compared non-diversified and diversified vector search for the same chunk.
* ✓ **Topic graph and GQL**: Walked from broad topics to their subtopics with GQL, and found example PDFs for each terminal topic with centroid searches.
* ✓ **Customer investigation**: Asked the topic knowledge base where to look, then answered Larkspur's complaint with a topic-aware knowledge source that cites the PDF evidence.

The topic graph tells you what the content covers and where to look. The source chunks are still the evidence.

➡️ Return to lab instructions.